In [ ]:
#!/usr/bin/env python3
import os
import re
import csv
import random
import pickle
import cv2
import numpy as np
import matplotlib.pyplot as plt
import shap
from scipy.stats import skew
from scipy.ndimage import center_of_mass, gaussian_filter
from skimage.filters import threshold_otsu
import tensorflow as tf
from tensorflow.keras.models import load_model
import sys

# Use tight layout for plots
plt.rcParams['figure.constrained_layout.use'] = True

# --------------------------------------------
#  Configuration (adjust paths as needed)
# --------------------------------------------
MODEL_PATH = (
    '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/'
    'polypclassificationmi/code/data/snapshots/all/'
    'hypVSadn_HDall2023_efficientnet_0_regularized0.0_256x256_'
    '1in_nf64_bnTrue_fcdo0.0_balancedTrue_loss_fl_gamma1.0_sgd_5fold0_best.h5'
)
SEQUENCES_FN = (
    '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/'
    'polypclassificationmi/data/degraded_image_list.txt'
)
OUTPUT_ROOT = (
    '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/'
    'polypclassificationmi/results/shap_artif'
)
EXPLAINER_PATH = os.path.join(OUTPUT_ROOT, 'shap_explainer.pkl')
BACKGROUND_SAMPLES = 50
INPUT_SIZE = (256, 256)  # (height, width) expected by the model


class ShapHeatmapCreator:
    def __init__(self,
                 model_path: str,
                 sequences_fn: str,
                 output_root: str,
                 explainer_path: str,
                 background_samples: int = 50,
                 input_size: tuple = (256, 256),
                 classes: list = [0, 1]):
        """
        model_path: path to the Keras .h5 model
        sequences_fn: txt file where each line is “degraded_full_path annotation_full_path label pid”
        output_root: directory where PNGs and CSV will be saved
        explainer_path: path to store/load the SHAP explainer (.pkl)
        background_samples: number of random masked crops for SHAP background
        input_size: (height, width) that the model expects
        classes: list of classes (not used directly here, but kept for compatibility)
        """
        self.model_path = model_path
        self.sequences_fn = sequences_fn
        self.output_root = output_root
        self.explainer_path = explainer_path
        self.background_samples = background_samples
        self.input_size = input_size
        self.classes = classes

        os.makedirs(self.output_root, exist_ok=True)

        # 1) Load the model (inference mode)
        self.model = load_model(self.model_path, compile=False)

        # 2) Prepare or load the SHAP explainer
        self.explainer = None
        self.background = None
        self._prepare_shap()

    def _prepare_shap(self):
        """
        Build or load a SHAP GradientExplainer with a masked-background batch.
        """
        # If explainer already exists, load it
        if os.path.exists(self.explainer_path):
            with open(self.explainer_path, 'rb') as f:
                self.explainer = pickle.load(f)
            return

        # Otherwise, build background batch and create the explainer
        background = self._build_background_batch(self.sequences_fn, self.background_samples)
        explainer = shap.GradientExplainer(self.model, background)

        # Save explainer to disk
        with open(self.explainer_path, 'wb') as f:
            pickle.dump(explainer, f)

        self.background = background
        self.explainer = explainer

    def _build_background_batch(self, file_list: str, n_samples: int):
        """
        Read random lines from the sequences file and extract masked crops
        to serve as background for SHAP. Returns an array of shape
        (n_samples, H_model, W_model, 3).
        """
        with open(file_list, 'r') as f:
            lines = f.readlines()
        random.shuffle(lines)

        samples = []
        for line in lines:
            parts = line.strip().split()
            # Expecting exactly 4 tokens: degraded_full_path, annotation_full_path, label, pid
            if len(parts) < 4:
                continue

            degraded_full_path = parts[0]
            ann_full_path = parts[1]
            # Skip if this is a brightGaussian_1.0 or brightGaussian_1.5 file
            filename = os.path.basename(degraded_full_path)
            if "brightGaussian_1.0.png" in filename or "brightGaussian_1.5.png" in filename:
                continue

            # Skip if paths don't exist
            if not (os.path.exists(degraded_full_path) and os.path.exists(ann_full_path)):
                continue

            # Preprocess and add to samples
            try:
                tensor, _, _, _ = self._preprocess(degraded_full_path, ann_full_path)
                samples.append(tensor)
                if len(samples) >= n_samples:
                    break
            except Exception:
                continue

        if not samples:
            raise RuntimeError("No valid background samples for SHAP!")

        # Concatenate into a single batch: shape (n_samples, H, W, 3)
        return np.concatenate(samples, axis=0)

    def _preprocess(self, img_path: str, ann_path: str, padding: float = 0.2):
        """
        1) Load image (RGB) and annotation mask (grayscale).
        2) Resize mask to image size and threshold to binary.
        3) Compute padded bounding box around the mask.
        4) Crop and resize to model input size, zeroing out pixels outside the mask.
        Returns:
          - tensor: shape (1, H_model, W_model, 3), dtype float32
          - mask_resized: boolean mask of shape (H_model, W_model)
          - bbox: (x0, y0, x1, y1) in the original full image
          - full_img: the full RGB image as a NumPy array (H_full, W_full, 3)
        """
        full_bgr = cv2.imread(img_path)
        if full_bgr is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        # Convert BGR to RGB
        full_img = cv2.cvtColor(full_bgr, cv2.COLOR_BGR2RGB)
        h_full, w_full = full_img.shape[:2]

        ann = cv2.imread(ann_path, cv2.IMREAD_GRAYSCALE)
        if ann is None:
            raise FileNotFoundError(f"Annotation not found: {ann_path}")

        # Binary mask (threshold at >10)
        mask_full = cv2.resize(ann, (w_full, h_full), interpolation=cv2.INTER_NEAREST) > 10

        # Find tight bounding box around mask
        ys, xs = np.where(mask_full)
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        box_w = x_max - x_min + 1
        box_h = y_max - y_min + 1

        pad_x = int(padding * box_w)
        pad_y = int(padding * box_h)
        x0 = max(0, x_min - pad_x)
        x1 = min(w_full, x_max + pad_x)
        y0 = max(0, y_min - pad_y)
        y1 = min(h_full, y_max + pad_y)

        # Crop image and mask
        crop_img = full_img[y0:y1, x0:x1]
        crop_mask = mask_full[y0:y1, x0:x1]

        # Resize both to model input size
        crop_resized = cv2.resize(crop_img, self.input_size, interpolation=cv2.INTER_LINEAR).astype('float32')
        mask_resized = cv2.resize(crop_mask.astype(np.uint8), self.input_size,
                                  interpolation=cv2.INTER_NEAREST).astype(bool)

        # Zero out pixels outside the mask (so SHAP focuses on ROI)
        crop_resized[~mask_resized] = 0.0

        # Add batch dimension
        tensor = np.expand_dims(crop_resized, axis=0)

        return tensor, mask_resized, (x0, y0, x1, y1), full_img

    def _compute_shap(self, crop_tensor: np.ndarray, mask_resized: np.ndarray):
        """
        Given a masked crop tensor, run the SHAP explainer + model to obtain:
          - heatmap: SHAP values averaged across channels, shape (H_model, W_model),
                     with zeros outside the mask
          - pred_class (int)
          - pred_prob (float)
        """
        # 1) Get SHAP values: returns a list of arrays (one per class)
        shap_vals = self.explainer.shap_values(crop_tensor)

        # 2) Run model inference for class + probability
        preds = self.model.predict(crop_tensor)
        pred_prob = float(np.max(preds, axis=-1)[0])
        pred_class = int(np.argmax(preds, axis=-1)[0])

        # 3) Select SHAP map for the predicted class, collapse channels by mean
        sv = shap_vals[pred_class][0]  # shape (H_model, W_model, C)
        heatmap = sv.mean(axis=-1)     # shape (H_model, W_model)

        # 4) Zero out outside the mask
        heatmap[~mask_resized] = 0.0

        return heatmap, pred_class, pred_prob

    def create_frames(self):
        """
        Iterate through every line in SEQUENCES_FN, perform:
          preprocess → compute_shap → compute metrics → generate a 1×4 figure →
          save PNGs in per-sequence folders → collect metrics →
        at the end, write a CSV with all metrics.
        """
        with open(self.sequences_fn, 'r') as f:
            entries = [line.strip().split() for line in f if line.strip()]

        results = []
        for parts in entries:
            # Expecting exactly 4 tokens: degraded_full_path, annotation_full_path, label, pid
            if len(parts) < 4:
                continue

            degraded_full_path = parts[0]
            ann_full_path = parts[1]
            label = int(parts[2])
            pid = parts[3]

            # Skip brightGaussian_1.0 and brightGaussian_1.5
            filename = os.path.basename(degraded_full_path)
            if "brightGaussian_1.0.png" in filename or "brightGaussian_1.5.png" in filename:
                continue

            if not (os.path.exists(degraded_full_path) and os.path.exists(ann_full_path)):
                print(f"Warning: missing {degraded_full_path} or {ann_full_path}, skipping")
                continue

            try:
                # 2) Preprocess (cropped + masked + resized)
                crop_tensor, mask_resized, (x0, y0, x1, y1), full_img = \
                    self._preprocess(degraded_full_path, ann_full_path)
                h_box = y1 - y0
                w_box = x1 - x0

                # 3) Compute SHAP heatmap on the masked crop
                raw_map, pred_class, pred_prob = self._compute_shap(crop_tensor, mask_resized)

                # 4) Upscale raw_map to the exact bounding box size
                heatmap_up = cv2.resize(raw_map, (w_box, h_box), interpolation=cv2.INTER_NEAREST)

                # 5) Create a full-size mask for that crop and zero out outside it
                full_mask = cv2.resize(mask_resized.astype(np.uint8), (w_box, h_box),
                                       interpolation=cv2.INTER_NEAREST).astype(bool)
                heatmap_up[~full_mask] = 0.0

                # 6) Load GT mask at full resolution and extract GT crop
                ann_full = cv2.imread(ann_full_path, cv2.IMREAD_GRAYSCALE)
                mask_full = cv2.resize(ann_full, (full_img.shape[1], full_img.shape[0]),
                                       interpolation=cv2.INTER_NEAREST) > 10
                gt_crop = mask_full[y0:y0 + h_box, x0:x0 + w_box]

                # 7) Smooth the heatmap, then Otsu threshold for positive / negative
                smooth_map = gaussian_filter(heatmap_up, sigma=1.0)

                th_pos = threshold_otsu(smooth_map)
                pos_mask = smooth_map >= th_pos

                neg_vals = -smooth_map[smooth_map < 0]
                th_neg = threshold_otsu(neg_vals) if neg_vals.size else 0.0
                neg_mask = smooth_map <= -th_neg

                # 8) IoU for positive and negative regions (percentage)
                overlap_p = np.logical_and(gt_crop, pos_mask).sum()
                union_p = np.logical_or(gt_crop, pos_mask).sum() + 1e-6
                iou_pos = overlap_p / union_p * 100.0

                overlap_n = np.logical_and(gt_crop, neg_mask).sum()
                union_n = np.logical_or(gt_crop, neg_mask).sum() + 1e-6
                iou_neg = overlap_n / union_n * 100.0

                # 9) Centers of mass (positive and negative)
                com_p = center_of_mass(pos_mask.astype(float))
                com_n = center_of_mass(neg_mask.astype(float))
                com_p_xy = (com_p[1], com_p[0]) if not np.isnan(com_p[0]) else (np.nan, np.nan)
                com_n_xy = (com_n[1], com_n[0]) if not np.isnan(com_n[0]) else (np.nan, np.nan)

                # 10) Overlay colors onto full image crop for pos/neg
                overlay = full_img.copy()
                patch = overlay[y0:y0 + h_box, x0:x0 + w_box]
                alpha = 0.5
                # Mark positive values in red
                patch[pos_mask] = ((1 - alpha) * patch[pos_mask] + alpha * np.array([255, 0, 0])).astype(np.uint8)
                # Mark negative values in blue
                patch[neg_mask] = ((1 - alpha) * patch[neg_mask] + alpha * np.array([0, 0, 255])).astype(np.uint8)
                overlay[y0:y0 + h_box, x0:x0 + w_box] = patch

                # 11) Image-quality metrics on the patch
                gray_patch = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
                lap_var = cv2.Laplacian(gray_patch, cv2.CV_64F).var()
                hsv_patch = cv2.cvtColor(patch, cv2.COLOR_RGB2HSV)
                hist = cv2.calcHist([hsv_patch], [2], None, [256], [0, 256]).flatten()
                skewness_value = skew(hist)

                # 12) Plot 1×4 and save figure
                fig, axs = plt.subplots(1, 4, figsize=(20, 5))
                axs[0].imshow(full_img)
                axs[0].axis('off')
                axs[0].set_title('Original Image')

                axs[1].imshow(gt_crop, cmap='gray')
                axs[1].axis('off')
                axs[1].set_title('GT Mask')

                axs[2].imshow(overlay)
                axs[2].axis('off')
                axs[2].set_title(f'SHAP Overlay\nIoU+:{iou_pos:.1f}%, IoU-:{iou_neg:.1f}%')

                axs[3].imshow(smooth_map, cmap='seismic')
                axs[3].axis('off')
                axs[3].set_title('SHAP Heatmap')

                seq = os.path.basename(os.path.normpath(os.path.dirname(degraded_full_path)))
                seq_dir = os.path.join(self.output_root, seq)
                os.makedirs(seq_dir, exist_ok=True)
                base = os.path.splitext(os.path.basename(degraded_full_path))[0]
                out_png = os.path.join(seq_dir, f"result_{base}.png")
                fig.savefig(out_png, dpi=200, bbox_inches='tight')
                plt.close(fig)

                # 13) Append metrics to results
                results.append({
                    'sequence':        seq,
                    'image':           base,
                    'predicted_class': pred_class,
                    'probability':     pred_prob,
                    'ground_truth':    label,
                    'iou_positive':    iou_pos,
                    'iou_negative':    iou_neg,
                    'laplacian_var':   lap_var,
                    'skewness':        skewness_value,
                    'com_pos_x':       com_p_xy[0],
                    'com_pos_y':       com_p_xy[1],
                    'com_neg_x':       com_n_xy[0],
                    'com_neg_y':       com_n_xy[1],
                })

            except Exception as e:
                print(f"Error processing {degraded_full_path}: {e}")
                continue

        # 14) Write final CSV
        csv_path = os.path.join(self.output_root, 'results.csv')
        if results:
            with open(csv_path, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=results[0].keys())
                writer.writeheader()
                writer.writerows(results)
            print(f"Results CSV saved to {csv_path}")
        else:
            print("No results to save to CSV.")


if __name__ == '__main__':
    shc = ShapHeatmapCreator(
        model_path=MODEL_PATH,
        sequences_fn=SEQUENCES_FN,
        output_root=OUTPUT_ROOT,
        explainer_path=EXPLAINER_PATH,
        background_samples=BACKGROUND_SAMPLES,
        input_size=INPUT_SIZE,
        classes=[0, 1]
    )
    shc.create_frames()


2025-06-05 10:45:33.266418: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib/python3.10/dist-packages/cv2/../../lib64:/usr/local/cuda/extras/CUPTI/lib64:/usr/local/cuda/compat/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/.singularity.d/libs
2025-06-05 10:45:33.266819: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib/python3.10/dist-packages/cv2/../../lib64:/usr/local/cuda/extras/CUPTI/lib64:/usr/local/cuda/compat/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/.singularity.d/libs
2025-06-05 10:45:33.267148: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcufft.so.10

INFO:tensorflow:Assets written to: ram://d9c444bc-d761-4e95-9f71-b030e2010330/assets


INFO:tensorflow:Assets written to: ram://d9c444bc-d761-4e95-9f71-b030e2010330/assets


INFO:tensorflow:Assets written to: ram://64a07b09-344a-4cff-ae31-74d883974ad6/assets


INFO:tensorflow:Assets written to: ram://64a07b09-344a-4cff-ae31-74d883974ad6/assets
`tf.keras.backend.set_learning_phase` is deprecated and will be removed after 2020-10-11. To update it, simply pass a True/False value to the `training` argument of the `__call__` method of your layer or model.


...

1/1 [==============================] - 0s 57ms/step


1/1 [==============================] - 0s 60ms/step


1/1 [==============================] - 0s 56ms/step


1/1 [==============================] - 0s 61ms/step


1/1 [==============================] - 0s 59ms/step
